# Multi-Port Vessel Loading Detection

Programmatic replacement for manually eyeballing Copernicus Browser screenshots — now covering multiple loading points, not just Zirku.

Pipeline:
1. Pull Sentinel-2 L2A true-color imagery for each AOI via the Copernicus Data Space Ecosystem (CDSE) Sentinel Hub API.
2. Build a rolling "clear water" background composite per AOI.
3. Diff each new scene against background, detect bright/elongated blobs (tankers), estimate length in meters.
4. Classify by size class (VLCC / Suezmax / Aframax) and log to a time-series table, per AOI.
5. Join the imagery detections against your existing Kpler port-call data on zone + date to flag AIS/imagery mismatches.

**AOIs configured so far** (see the config cell below — add more the same way):
- Das SPM 1 & 2, Zirku SPM 1 & 2, Um Lulu SPM — UAE offshore polygons (`single_spm_polygon` template)
- Yanbu (King Fahd Industrial Port, North Crude Terminal jetty): ~23.94, 38.26
- Muajjiz, CPC (Crimea) — polygon + berth slots

**Important caveat on Fujairah anchorage and any diffuse zone**: unlike an SPM or a jetty, an anchorage is a large area where vessels sit spread out and idle, not just load/discharge. A background-diff "is there a new blob" approach works fine for a tight berth AOI, but for an anchorage you're really doing *vessel counting/tracking* across a wide area, which is a different (harder, noisier) problem than loading detection at a point. Recommend treating anchorage AOIs separately from berth/SPM AOIs in the analysis — don't expect the same precision.

**Best practice for AOI definitions**: since you already have Kpler port-call/zone data, the most reliable way to define each AOI is to **pull the same zone geometry Kpler uses internally** (if exposed via their API/data export) rather than hand-picking a lat/lon + buffer here. That guarantees your imagery detections and your AIS port-calls are keyed to the *same* physical zone, which matters a lot for the join in the last section. Where you don't have that, a point + buffer (as below) is a reasonable approximation for tight berths/SPMs — just not for anchorages.

**Requirements**
```
pip install sentinelhub opencv-python-headless numpy pandas matplotlib --break-system-packages
```

**Credentials**: register a free account at https://dataspace.copernicus.eu, create an OAuth client (Dashboard > User Settings), then set `SH_CLIENT_ID` / `SH_CLIENT_SECRET` as environment variables (or fill in below — don't commit them).

In [1]:
import os
import datetime as dt

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sentinelhub import (
    SHConfig,
    BBox,
    CRS,
    Geometry,
    bbox_to_dimensions,
    SentinelHubRequest,
    SentinelHubCatalog,
    DataCollection,
    MimeType,
)


from pathlib import Path
import sys

# Allow imports from this folder when running in Jupyter.
NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR / "vessel_crossing" / "port_loading_gaps", NOTEBOOK_DIR / "port_loading_gaps"]:
    if (candidate / "vessel_detect.py").exists():
        NOTEBOOK_DIR = candidate
        break
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from vessel_detect import detect_for_aoi, detect_berth_slots, detect_s1_vessels, build_berth_slot_masks, diagnose_berth_slots
from aoi_config import (
    TERMINAL_TEMPLATES,
    apply_template,
    get_detect_params_from_cfg,
    print_setup_checklist,
    validate_terminal,
    format_aoi_entry_python,
    placeholder_berth_slots,
)


BASE = NOTEBOOK_DIR.parent
ENV_PATH = BASE / ".env"


from dotenv import load_dotenv

load_dotenv(ENV_PATH)

COPERNICUS_CLIENT = os.getenv("SH_CLIENT_ID")
COPERNICUS_SECRET = os.getenv("SH_CLIENT_SECRET")



c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\.venv\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
COPERNICUS_CLIENT

'sh-f6be6f09-7f86-45d8-ad8c-26638d079b6a'

In [3]:
# --- Credentials -------------------------------------------------------
config = SHConfig()
config.sh_client_id = COPERNICUS_CLIENT
config.sh_client_secret = COPERNICUS_SECRET
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

# Process API uses each DataCollection's service_url, not sh_base_url alone.
# Default SENTINEL2_L2A points at commercial Sentinel Hub; override for CDSE.
S2_L2A_CDSE = DataCollection.SENTINEL2_L2A.define_from(
    "s2l2a", service_url=config.sh_base_url
)
S1_IW_CDSE = DataCollection.SENTINEL1_IW.define_from(
    "s1iw", service_url=config.sh_base_url
)

if not config.sh_client_id or not config.sh_client_secret:
    print("WARNING: set SH_CLIENT_ID / SH_CLIENT_SECRET env vars before running requests.")

In [4]:
# UAE SPM polygons are defined in the AOI definitions cell below (DAS_SPM*, ZIRKU_SPM*, UM_LULU_SPM*).

#zirku_1 = {"type":"Polygon","coordinates":[[[52.993498,25.005817],[52.976246,25.005817],[52.976246,25.01624],[52.993498,25.01624],[52.993498,25.005817]]]}
#zirku_box = {"type":"Polygon","coordinates":[[[53.004913,25.030939],[52.98912,25.003872],[52.976503,25.009084],[52.992125,25.035839],[53.004913,25.030939]]]}



In [5]:
# --- AOI definitions ------------------------------------------------------
# New terminal? Use template="berth_slots_jetty" or "single_spm" — see checklist cell below.
# Do NOT copy the full detect dict; override individual keys only after Kpler compare.
# Location types:
#   "point"   -> center lat/lon + buffer_m (tight SPM / single berth)
#   "bbox"    -> (min_lon, min_lat, max_lon, max_lat)
#   "polygon" -> GeoJSON polygon from Copernicus Browser (best for multi-berth sites)
#
# Optional: mode = "single_berth" (default) or "multi_berth" (count vessels in a zone)

YANBU_POLYGON = {
    "type": "Polygon",
    "coordinates": [
        [
            [38.221478, 23.945802],
            [38.239846, 23.936839],
            [38.237851, 23.933348],
            [38.219182, 23.942566],
            [38.221478, 23.945802],
        ]
    ],
}

MUAJJIZ_POLYGON = {"type":"Polygon",
    "coordinates":
    [
        [
            [38.395264,23.810083],
            [38.398311,23.812399],
            [38.410499,23.801935],
            [38.407516,23.799462],
            [38.395264,23.810083]
            
        ]
    ]
        
}

CPC_POLYGON = {"type":"Polygon",
    "coordinates":
    [
        [
            [37.599936,44.637086],
            [37.666883,44.628779],
            [37.663364,44.617722],
            [37.595644,44.628596],
            [37.599936,44.637086]
        ]
    ]
}



DAS_SPM1_POLYGON = {"type": "Polygon", "coordinates": [[[52.937965, 25.129744], [52.921314, 25.129744], [52.921314, 25.140934], [52.937965, 25.140934], [52.937965, 25.129744]]]}
DAS_SPM2_POLYGON = {"type": "Polygon", "coordinates": [[[53.001223, 25.072306], [53.023195, 25.072306], [53.023195, 25.061732], [53.001223, 25.061732], [53.001223, 25.072306]]]}
ZIRKU_SPM1_POLYGON = {"type":"Polygon", "coordinates":[[[52.974787,25.005195], [52.974787,25.016862], [52.996759,25.016862], [52.996759,25.005195], [52.974787,25.005195]]]}
ZIRKU_SPM2_POLYGON = {"type": "Polygon", "coordinates": [[[52.981138, 25.023084], [53.002596, 25.023084], [53.002596, 25.033972], [52.981138, 25.033972], [52.981138, 25.023084]]]}
UM_LULU_SPM_POLYGON = {"type": "Polygon", "coordinates": [[[53.004055, 25.061888], [53.004055, 25.073005], [53.021908, 25.073005], [53.021908, 25.061888], [53.004055, 25.061888]]]}

# Pixel-space berth boxes [x1, y1, x2, y2] — tune with the picker cell below.
YANBU_BERTH_SLOTS = [
    {"name": "berth_1", "box": [15, 20, 68, 45]},
    {"name": "berth_2", "box": [60, 47, 101, 75]},
    {"name": "berth_3", "box": [102, 70, 144, 101]},
    {"name": "berth_4", "box": [144, 90, 198, 119]},
]

MUAJJIZ_BERTH_SLOTS = [
    {"name": "berth_1", "box": [16, 16, 61, 50]},
    {"name": "berth_2", "box": [51, 55, 95, 86]},
    {"name": "berth_3", "box": [83, 90, 131, 124]},
]

CPC_BERTH_SLOTS = [
    {"name": "spm_left", "box": [49, 47, 145, 105]},  # placeholders — run picker
    {"name": "spm_center", "box": [271, 53, 349, 91]},
    {"name": "spm_right", "box": [485, 129, 550, 173]},
]

# One slot per SPM polygon — run picker on each AOI to tighten boxes.
DAS_SPM1_SLOTS = [{"name": "spm", "box": [20, 20, 120, 120]}]
DAS_SPM2_SLOTS = [{"name": "spm", "box": [69, 25, 196, 88]}]
ZIRKU_SPM1_SLOTS = [{"name": "spm", "box": [20, 20, 120, 120]}]
ZIRKU_SPM2_SLOTS = [{"name": "spm", "box": [20, 20, 120, 120]}]
UM_LULU_SPM_SLOTS = [{"name": "spm", "box": [50, 17, 149, 104]}]

AOI_CONFIG = {
    "das_spm_1": {
        "type": "polygon",
        "geojson": DAS_SPM1_POLYGON,
        "template": "single_spm_polygon",
        "berth_slots_px": DAS_SPM1_SLOTS,
    },
    "das_spm_2": {
        "type": "polygon",
        "geojson": DAS_SPM2_POLYGON,
        "template": "single_spm_polygon",
        "berth_slots_px": DAS_SPM2_SLOTS,
    },
    "zirku_spm_1": {
        "type": "polygon",
        "geojson": ZIRKU_SPM1_POLYGON,
        "template": "single_spm_polygon",
        "berth_slots_px": ZIRKU_SPM1_SLOTS,
    },
    "zirku_spm_2": {
        "type": "polygon",
        "geojson": ZIRKU_SPM2_POLYGON,
        "template": "single_spm_polygon",
        "berth_slots_px": ZIRKU_SPM2_SLOTS,
    },
    "um_lulu_spm": {
        "type": "polygon",
        "geojson": UM_LULU_SPM_POLYGON,
        "template": "single_spm_polygon",
        "berth_slots_px": UM_LULU_SPM_SLOTS,
    },
    "yanbu_north_crude_terminal": {
        "type": "polygon",
        "geojson": YANBU_POLYGON,
        "template": "berth_slots_jetty",
        "berth_slots_px": YANBU_BERTH_SLOTS,
    },
    "muajjiz": {
        "type": "polygon",
        "geojson": MUAJJIZ_POLYGON,
        "template": "berth_slots_jetty",
        "berth_slots_px": MUAJJIZ_BERTH_SLOTS,
    },
    "cpc": {
        "type": "polygon",
        "geojson": CPC_POLYGON,
        "template": "berth_slots_spm",
        "berth_slots_px": CPC_BERTH_SLOTS,
    },
    #"fujairah_oil_terminal": {"type": "point", "lat": 25.188, "lon": 56.360, "buffer_m": 1800},
    #"fujairah_anchorage": {"type": "bbox", "bbox": (56.40, 25.05, 56.65, 25.25)},
}

RESOLUTION = 10  # meters/pixel, native S2 RGB bands
MAX_REQUEST_PX = 2500  # Sentinel Hub Process API width/height cap

def buffer_point(lat, lon, meters):
    """Rough square buffer around a point in degrees (good enough at these latitudes)."""
    dlat = meters / 111_000
    dlon = meters / (111_000 * np.cos(np.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def resolve_geometry(cfg):
    if cfg["type"] == "polygon":
        return Geometry(cfg["geojson"], crs=CRS.WGS84)
    return None

def resolve_bbox(cfg, geometry=None):
    if geometry is not None:
        return geometry.bbox
    if cfg["type"] == "point":
        return BBox(bbox=buffer_point(cfg["lat"], cfg["lon"], cfg["buffer_m"]), crs=CRS.WGS84)
    if cfg["type"] == "bbox":
        return BBox(bbox=cfg["bbox"], crs=CRS.WGS84)
    raise ValueError(f"Unknown AOI type: {cfg['type']}")

def bbox_to_request_size(bbox, resolution, max_px=MAX_REQUEST_PX):
    """Return (width, height) capped for Process API, plus effective m/px."""
    w, h = bbox_to_dimensions(bbox, resolution=resolution)
    if w <= max_px and h <= max_px:
        return (w, h), resolution
    scale = min(max_px / w, max_px / h)
    return (max(1, int(w * scale)), max(1, int(h * scale))), resolution / scale

geometries = {}
bboxes = {}
sizes = {}
resolutions = {}
for name, cfg in AOI_CONFIG.items():
    geom = resolve_geometry(cfg)
    geometries[name] = geom
    bbox = resolve_bbox(cfg, geom)
    bboxes[name] = bbox
    sizes[name], resolutions[name] = bbox_to_request_size(bbox, RESOLUTION)
    if resolutions[name] > RESOLUTION:
        print(
            f"{name}: downscaled to {sizes[name]} px "
            f"(effective resolution {resolutions[name]:.1f} m/px)"
        )

LOADING_AOIS = [name for name in AOI_CONFIG if name != "fujairah_anchorage"]

# Detection tuning profiles — shared algorithm, different strictness per site type.
DETECT_PROFILES = {
    "single_berth": {
        "diff_thresh": 40,
        "min_aspect": 2.5,
        "min_length_m": 100,
        "max_length_m": 400,
        "min_area_px": 20,
        "use_saturation": False,
        "sat_thresh": 60,
        "max_blob_frac": 0.15,
        "water_gray_lo": 8,
        "water_gray_hi": 80,
        "dedupe_px": 20,
    },
    "multi_berth": {
        "diff_thresh": 25,
        "use_adaptive_diff": True,
        "diff_percentile": 90,
        "min_aspect": 1.65,
        "min_length_m": 110,
        "max_length_m": 350,
        "min_area_px": 35,
        "use_saturation": False,
        "use_hull_colors": True,
        "sat_thresh": 55,
        "max_blob_frac": 0.15,
        "water_gray_lo": 8,
        "water_gray_hi": 85,
        "dedupe_px": 50,
        "hull_min_sep_px": 60,
    },
}


def get_detect_params(aoi_name):
    return get_detect_params_from_cfg(AOI_CONFIG[aoi_name], DETECT_PROFILES)


def resolved_aoi_config(aoi_name):
    """AOI config with template merged (mode, sensor, detect defaults)."""
    return apply_template(AOI_CONFIG[aoi_name])


detect_params_by_aoi = {name: get_detect_params(name) for name in AOI_CONFIG}

# Runtime config: template merged into mode / sensor / detect
RESOLVED_AOI_CONFIG = {n: apply_template(c) for n, c in AOI_CONFIG.items()}

sizes

{'das_spm_1': (166, 126),
 'das_spm_2': (220, 120),
 'zirku_spm_1': (220, 132),
 'zirku_spm_2': (215, 124),
 'um_lulu_spm': (178, 126),
 'yanbu_north_crude_terminal': (211, 137),
 'muajjiz': (156, 143),
 'cpc': (569, 206)}

## Adding a new terminal

You do **not** need to copy a full `detect` dict from another site. Pick a **template** and only add overrides after Kpler compare shows errors.

| Template | Use for | You must provide |
|----------|---------|------------------|
| `berth_slots_spm` | CPC — multiple SPMs in open water | `geojson`, tight `berth_slots_px` |
| `berth_slots_jetty` | Yanbu, Muajjiz — jetty berths | `geojson`, `berth_slots_px` (picker) |
| `single_spm_polygon` | Das / Zirku / Um Lulu — one SPM per polygon | `geojson`, one tight `berth_slots_px` |
| `single_spm` | Legacy point+buffer SPM (prefer polygon) | `lat`, `lon`, `buffer_m` |
| `multi_berth_blob` | Legacy blob counting in a polygon | `geojson` |

**Minimal CPC-style entry:**
```python
"cpc": {
    "type": "polygon",
    "geojson": CPC_POLYGON,
    "template": "berth_slots_spm",
    "berth_slots_px": CPC_BERTH_SLOTS,
},
```

Run the checklist cell below after every new AOI. Tune `detect={...}` only on mismatch dates.

In [6]:
# --- New terminal checklist (change SETUP_AOI and re-run) -------------------

SETUP_AOI = "cpc"  # key in AOI_CONFIG

print_setup_checklist(SETUP_AOI, AOI_CONFIG[SETUP_AOI])

# Resolved params after template merge (what detection actually uses)
pd.Series(get_detect_params(SETUP_AOI)).sort_index()

# Blank snippet for the next terminal:
# print(format_aoi_entry_python(
#     "my_terminal",
#     template="berth_slots_jetty",
#     geojson_var="MY_POLYGON",
#     slots_var="MY_BERTH_SLOTS",
# ))

# Placeholder slots until picker is run (3 SPMs example):
# placeholder_berth_slots(3, prefix="spm")

=== Setup checklist: cpc ===

Template: berth_slots_spm
  Several SPMs in one AOI. Shrink boxes to the vessel footprint on water only.

Steps:
  1. Draw polygon covering all SPMs
  2. Pick tight boxes on each SPM mooring (water + vessel only, not wide wake)
  3. Use template='berth_slots_spm'
  4. Run diagnose_berth_slots() on known occupied/empty dates
  5. Run Kpler compare; override slot_* only if needed

Tunable detect keys (only if Kpler compare shows mismatches):
  • slot_relative_blob_ratio — raise to kill empty-SPM speckle FPs (try 0.45–0.5)
  • slot_hull_frac — lower if dark-green hulls missed (spm_left)
  • slot_min_blob_frac — raise to reduce FPs; lower to catch pale hulls
  • slot_water_only — keep True for SPMs (ignore non-water pixels in box)
  • use_ndwi_water / ndwi_water_min — NDWI (B03−B08) water mask; try 0.45 if glint shrinks valid water

Validation:
  [ok] Required fields present

Kpler hint: LocationSpec("???", kind="zone")  # confirm CPC zone name in Kpler

No de

dedupe_px                      50
diff_percentile                88
diff_thresh                    25
hull_min_sep_px                60
max_blob_frac                0.15
max_length_m                  350
min_area_px                    35
min_aspect                   1.65
min_length_m                  110
ndwi_water_min                0.5
sat_thresh                     55
slot_diff_abs                  40
slot_diff_blob_frac          0.16
slot_diff_frac               0.18
slot_hull_frac              0.025
slot_min_blob_frac           0.32
slot_min_hull_blob_frac      0.02
slot_relative_blob_ratio      0.4
slot_use_relative            True
slot_water_only              True
use_adaptive_diff            True
use_hull_colors              True
use_ndwi_water               True
use_saturation              False
water_gray_hi                 100
water_gray_lo                   8
dtype: object

In [7]:
# --- Catalog search: find available clear-ish scenes over a date range ---

def _aoi_bounds_kwargs(aoi_name):
    geom = geometries[aoi_name]
    return {"geometry": geom} if geom is not None else {"bbox": bboxes[aoi_name]}


def search_scenes(aoi_name, start, end, max_cloud=30):
    catalog = SentinelHubCatalog(config=config)
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        **_aoi_bounds_kwargs(aoi_name),
        time=(start, end),
        filter=f"eo:cloud_cover < {max_cloud}",
        fields={"include": ["id", "properties.datetime", "properties.eo:cloud_cover"], "exclude": []},
    )
    results = list(search_iterator)
    dates = sorted({r["properties"]["datetime"][:10] for r in results})
    return dates

# Example:
# dates = search_scenes("yanbu_north_crude_terminal", "2026-07-25", "2026-08-03")
# dates

In [8]:
# --- Evalscript: true color + NDWI (B03/B08), cloud-masked -----------------

CACHE_DIR = NOTEBOOK_DIR / ".cache" / "sentinel"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EVALSCRIPT_RGB_NDWI = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["B04", "B03", "B02", "B08", "SCL"] }],
    output: [
      { id: "rgb", bands: 3, sampleType: "UINT8" },
      { id: "ndwi", bands: 1, sampleType: "FLOAT32" }
    ]
  };
}
function evaluatePixel(sample) {
  // SCL 3 = cloud shadow, 8/9 = cloud medium/high, 10 = thin cirrus
  if ([3, 8, 9, 10].includes(sample.SCL)) {
    return { rgb: [0, 0, 0], ndwi: [-999] };
  }
  var ndwi = (sample.B03 - sample.B08) / (sample.B03 + sample.B08 + 1e-6);
  return {
    rgb: [
      Math.min(255, sample.B04 * 255 * 3.5),
      Math.min(255, sample.B03 * 255 * 3.5),
      Math.min(255, sample.B02 * 255 * 3.5)
    ],
    ndwi: [ndwi]
  };
}
"""


def _fetch_s2_rgb_ndwi(aoi_name, date_str, use_cache=True):
    """Fetch true-color RGB and McFeeters NDWI in one Sentinel Hub request."""
    cache_dir = CACHE_DIR / "s2" / aoi_name
    rgb_path = cache_dir / f"{date_str}.npy"
    ndwi_path = cache_dir / f"{date_str}_ndwi.npy"

    if use_cache and rgb_path.exists() and ndwi_path.exists():
        return np.load(rgb_path), np.load(ndwi_path)

    request = SentinelHubRequest(
        evalscript=EVALSCRIPT_RGB_NDWI,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=S2_L2A_CDSE,
                time_interval=(date_str, date_str),
                mosaicking_order="leastCC",
            )
        ],
        responses=[
            SentinelHubRequest.output_response("rgb", MimeType.PNG),
            SentinelHubRequest.output_response("ndwi", MimeType.TIFF),
        ],
        size=sizes[aoi_name],
        config=config,
        **_aoi_bounds_kwargs(aoi_name),
    )
    data = request.get_data()
    if not data:
        return None, None

    # Multi-output responses come back as one dict: {"rgb.png": array, "ndwi.tif": array}
    if isinstance(data[0], dict):
        payload = data[0]
        rgb = np.asarray(payload["rgb.png"])
        ndwi = np.asarray(payload["ndwi.tif"], dtype=np.float32)
    else:
        rgb = np.asarray(data[0])
        ndwi = np.asarray(data[1], dtype=np.float32)
        if ndwi.ndim == 3:
            ndwi = ndwi[..., 0]
    ndwi = np.where(ndwi <= -900, np.nan, ndwi)

    if use_cache:
        cache_dir.mkdir(parents=True, exist_ok=True)
        np.save(rgb_path, rgb)
        np.save(ndwi_path, ndwi)
    return rgb, ndwi


def fetch_scene(aoi_name, date_str, use_cache=True):
    """Fetch a single-day true-color image (RGB uint8 array) for an AOI."""
    rgb, _ = _fetch_s2_rgb_ndwi(aoi_name, date_str, use_cache=use_cache)
    return rgb


def fetch_ndwi(aoi_name, date_str, use_cache=True):
    """Fetch NDWI float32 array (cached alongside RGB)."""
    _, ndwi = _fetch_s2_rgb_ndwi(aoi_name, date_str, use_cache=use_cache)
    return ndwi


# --- Sentinel-1 SAR (VV backscatter, dB) -----------------------------------

EVALSCRIPT_S1_VV = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ["VV"], units: "LINEAR_POWER" }],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}
function evaluatePixel(sample) {
  return [10 * Math.log(sample.VV) / Math.LN10];
}
"""


def search_s1_scenes(aoi_name, start, end):
    catalog = SentinelHubCatalog(config=config)
    search_iterator = catalog.search(
        DataCollection.SENTINEL1_IW,
        **_aoi_bounds_kwargs(aoi_name),
        time=(start, end),
        fields={"include": ["id", "properties.datetime"], "exclude": []},
    )
    return sorted({r["properties"]["datetime"][:10] for r in search_iterator})


def fetch_s1_vv(aoi_name, date_str, window_days=1, use_cache=True):
    """Fetch Sentinel-1 VV backscatter (dB) for a date window."""
    cache_path = CACHE_DIR / "s1" / aoi_name / f"{date_str}.npy"
    if use_cache and cache_path.exists():
        return np.load(cache_path)
    start = date_str
    end_dt = dt.datetime.strptime(date_str, "%Y-%m-%d") + dt.timedelta(days=window_days)
    end = end_dt.strftime("%Y-%m-%d")
    request = SentinelHubRequest(
        evalscript=EVALSCRIPT_S1_VV,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=S1_IW_CDSE,
                time_interval=(start, end),
                mosaicking_order="mostRecent",
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        size=sizes[aoi_name],
        config=config,
        **_aoi_bounds_kwargs(aoi_name),
    )
    data = request.get_data()
    if not data:
        return None
    arr = data[0]
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.float32)
    if use_cache and np.isfinite(arr).any():
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, arr)
    return arr


def get_acquisition_dates(aoi_name, start, end, max_cloud=30):
    """Union of S2 (clear) and S1 dates depending on AOI sensor config."""
    cfg = RESOLVED_AOI_CONFIG[aoi_name]
    sensor = cfg.get("sensor", "s2")
    dates = set()
    if sensor in ("s2", "s2_s1"):
        dates.update(search_scenes(aoi_name, start, end, max_cloud=max_cloud))
    if sensor in ("s1", "s2_s1"):
        dates.update(search_s1_scenes(aoi_name, start, end))
    return sorted(dates)

In [9]:
# --- Background composite (median of several clear scenes, no ships) ----

def build_background(aoi_name, dates_for_background, exclude_dates=None):
    exclude = set(exclude_dates or [])
    stack = []
    skipped = []
    for d in dates_for_background:
        if d in exclude:
            skipped.append(d)
            continue
        img = fetch_scene(aoi_name, d)
        if img is not None:
            stack.append(img.astype(np.float32))
    if not stack:
        raise RuntimeError(
            f"No scenes retrieved for background composite ({aoi_name}). "
            f"Candidates: {list(dates_for_background)}; "
            f"skipped (overlap with timeseries): {skipped or list(exclude)[:8]}. "
            "Use a bg_window before the analysis range (e.g. June), not the same dates."
        )
    return np.median(np.stack(stack, axis=0), axis=0).astype(np.uint8)


def get_background(
    aoi_name,
    bg_window=("2026-06-01", "2026-06-30"),
    max_scenes=4,
    force_refresh=False,
    use_cache=True,
):
    """Build a ship-sparse median background for one AOI (cached to disk)."""
    bg_path = CACHE_DIR / "bg" / aoi_name / "median.npy"
    if use_cache and not force_refresh and bg_path.exists():
        print(f"Loaded cached background: {aoi_name}")
        return np.load(bg_path)
    bg_dates = search_scenes(aoi_name, *bg_window)[:max_scenes]
    print(f"Fetching background for {aoi_name} ({len(bg_dates)} scenes)...")
    bg = build_background(aoi_name, bg_dates)
    if use_cache:
        bg_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(bg_path, bg)
    return bg


backgrounds = {}  # lazy — populated by ensure_background()


def ensure_background(aoi_name, force_refresh=False, **kwargs):
    """Load or build background for one AOI only when you need it."""
    if force_refresh or aoi_name not in backgrounds:
        backgrounds[aoi_name] = get_background(aoi_name, force_refresh=force_refresh, **kwargs)
    return backgrounds[aoi_name]


# Optional pre-warm (uncomment the AOI you're debugging — ~30-60s first time, instant after):
# ensure_background("muajjiz")

In [10]:
# --- Ship detection: water-masked diff + rotated bounding boxes -----------

SIZE_CLASSES = [
    (300, 380, "VLCC"),
    (250, 300, "Suezmax"),
    (220, 250, "Aframax"),
    (0, 220, "smaller/unclassified"),
]


def classify_length(length_m):
    for lo, hi, label in SIZE_CLASSES:
        if lo <= length_m < hi:
            return label
    return "unclassified"


def water_mask(rgb, gray_lo=8, gray_hi=80):
    """Open-water pixels: dark enough to be water, not SCL-black or bright jetty."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return ((gray > gray_lo) & (gray < gray_hi)).astype(np.uint8)


def aoi_valid_mask(rgb, black_thresh=5):
    """Non-masked polygon pixels (evalscript returns black outside AOI / on clouds)."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return (gray > black_thresh).astype(np.uint8)


def _validate_scene_pair(scene, background, label="scene/background"):
    if scene.shape != background.shape:
        raise ValueError(
            f"{label} shape mismatch: scene {scene.shape[:2]} vs background {background.shape[:2]}. "
            "Fetch the scene and build the background for the same AOI "
            "(use backgrounds[aoi_name] from the background cell)."
        )


def build_change_mask(scene, background, params):
    """Diff scene vs background on water; optionally add saturated hull cue."""
    gray_scene = cv2.cvtColor(scene, cv2.COLOR_RGB2GRAY)
    gray_bg = cv2.cvtColor(background, cv2.COLOR_RGB2GRAY)
    wm = water_mask(scene, params["water_gray_lo"], params["water_gray_hi"])

    diff = cv2.absdiff(gray_scene, gray_bg)
    diff[~wm.astype(bool)] = 0

    if params.get("use_adaptive_diff"):
        # Fixed thresholds fail when global water brightness shifts between dates;
        # keep the top-N% of water-pixel changes instead (multi-berth sites).
        vals = diff[diff > 0]
        if len(vals) == 0:
            combined = np.zeros_like(diff, dtype=np.uint8)
        else:
            t = float(np.percentile(vals, params.get("diff_percentile", 90)))
            combined = (diff >= t).astype(np.uint8) * 255
    else:
        _, combined = cv2.threshold(diff, params["diff_thresh"], 255, cv2.THRESH_BINARY)

    if params.get("use_saturation") and not params.get("use_adaptive_diff"):
        sat = cv2.cvtColor(scene, cv2.COLOR_RGB2HSV)[:, :, 1]
        _, sat_mask = cv2.threshold(sat, params["sat_thresh"], 255, cv2.THRESH_BINARY)
        sat_mask = cv2.bitwise_and(sat_mask, wm * 255)
        weak_diff = (diff > max(params["diff_thresh"] // 2, 10)).astype(np.uint8) * 255
        sat_mask = cv2.bitwise_and(sat_mask, weak_diff)
        combined = cv2.bitwise_or(combined, sat_mask)

    kernel = np.ones((3, 3), np.uint8)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=2)
    return combined, wm


def hull_color_mask(scene, valid):
    """Scene-only cue for painted hulls; uses AOI valid pixels, not strict water mask.

    Bright orange/red tankers (e.g. Muajjiz) exceed water_gray_hi and were previously
    excluded when hull detection reused the water mask.
    """
    r, g, b = scene[:, :, 0], scene[:, :, 1], scene[:, :, 2]
    gs = cv2.cvtColor(scene, cv2.COLOR_RGB2GRAY)
    base = valid.astype(bool)
    orange = (r > 120) & (r > g * 1.25) & (r > b * 1.15) & base
    red = (r > 80) & (r > g * 1.08) & (r > b * 1.05) & (gs < 140) & base
    olive = (g > 35) & (g > r * 1.0) & (g > b * 1.0) & (gs > 15) & (gs < 130) & base
    mask = (orange | red | olive).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)
    return mask


def _contours_to_detections(contours, params, resolution, max_blob_area):
    detections = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < params["min_area_px"] or area > max_blob_area:
            continue

        (_, _), (rw, rh), angle = cv2.minAreaRect(cnt)
        length_px = max(rw, rh)
        width_px = min(rw, rh)
        aspect = length_px / max(width_px, 1)
        if aspect < params["min_aspect"]:
            continue

        length_m = length_px * resolution
        width_m = width_px * resolution
        if length_m < params["min_length_m"] or length_m > params["max_length_m"]:
            continue

        m = cv2.moments(cnt)
        cx = m["m10"] / m["m00"] if m["m00"] else 0.0
        cy = m["m01"] / m["m00"] if m["m00"] else 0.0

        detections.append(
            {
                "length_m": round(length_m, 1),
                "width_m": round(width_m, 1),
                "size_class": classify_length(length_m),
                "centroid_xy": (round(cx, 1), round(cy, 1)),
                "angle_deg": round(float(angle), 1),
            }
        )
    return detections


def detect_vessels(scene, background, resolution=RESOLUTION, detect_params=None):
    """
    Returns list of dicts: length_m, width_m, size_class, centroid_xy, angle_deg.
    Uses minAreaRect so diagonal tankers at jetties are not rejected by axis-aligned filters.
    """
    params = detect_params or DETECT_PROFILES["single_berth"]
    _validate_scene_pair(scene, background)
    mask, wm = build_change_mask(scene, background, params)
    water_px = max(int(wm.sum()), 1)
    max_blob_area = int(water_px * params["max_blob_frac"])

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = _contours_to_detections(contours, params, resolution, max_blob_area)

    if params.get("use_hull_colors"):
        valid = aoi_valid_mask(scene)
        hull_contours, _ = cv2.findContours(
            hull_color_mask(scene, valid), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        hull_params = {
            **params,
            "min_aspect": params.get("hull_min_aspect", params["min_aspect"]),
            "min_length_m": params.get("hull_min_length_m", params["min_length_m"]),
        }
        hull_dets = _contours_to_detections(hull_contours, hull_params, resolution, max_blob_area)
        min_sep = params.get("hull_min_sep_px", 45)
        for hd in hull_dets:
            hx, hy = hd["centroid_xy"]
            if all(np.hypot(hx - d["centroid_xy"][0], hy - d["centroid_xy"][1]) >= min_sep for d in detections):
                detections.append(hd)

    dedupe_px = params.get("dedupe_px", 20)
    deduped = []
    for det in sorted(detections, key=lambda d: -d["length_m"]):
        cx, cy = det["centroid_xy"]
        if any(np.hypot(cx - d["centroid_xy"][0], cy - d["centroid_xy"][1]) < dedupe_px for d in deduped):
            continue
        deduped.append(det)
    return deduped


def detect_vessels_for_aoi(aoi_name, scene, background, s1_vv=None, ndwi=None):
    """Dispatch to blob, berth-slot, or SAR detection based on AOI config."""
    return detect_for_aoi(
        scene,
        background,
        RESOLVED_AOI_CONFIG[aoi_name],
        resolutions[aoi_name],
        detect_params_by_aoi[aoi_name],
        s1_vv_db=s1_vv,
        ndwi=ndwi,
    )

In [11]:
# --- Full loop: build time series for one AOI over a date range ---------

def build_timeseries(aoi_name, start, end, background_dates, max_cloud=30):
    cfg = RESOLVED_AOI_CONFIG[aoi_name]
    sensor = cfg.get("sensor", "s2")
    scene_dates = get_acquisition_dates(aoi_name, start, end, max_cloud=max_cloud)
    background = None
    if cfg.get("mode") != "single_berth" or sensor != "s1":
        if sensor != "s1":
            s2_dates = set(search_scenes(aoi_name, start, end, max_cloud=max_cloud))
            bg_dates = [d for d in background_dates if d not in s2_dates]
            if bg_dates:
                background = build_background(aoi_name, bg_dates, exclude_dates=s2_dates)
            else:
                # bg_window overlapped the timeseries — fall back to cached pre-period composite
                background = ensure_background(aoi_name)

    rows = []
    for d in scene_dates:
        used_sensor = "s2"
        scene = None
        ndwi = None
        if sensor != "s1":
            scene, ndwi = _fetch_s2_rgb_ndwi(aoi_name, d)
        s1_vv = None
        dets = []

        if scene is not None and sensor in ("s2", "s2_s1") and background is not None:
            dets = detect_vessels_for_aoi(aoi_name, scene, background, ndwi=ndwi)
            used_sensor = "s2"
        elif sensor in ("s1", "s2_s1"):
            s1_vv = fetch_s1_vv(aoi_name, d)
            if s1_vv is not None and np.isfinite(s1_vv).any():
                dets = detect_s1_vessels(s1_vv, resolutions[aoi_name], detect_params_by_aoi[aoi_name])
                used_sensor = "s1"

        if dets:
            for det in dets:
                rows.append({
                    "aoi": aoi_name,
                    "date": d,
                    "vessel_count": len(dets),
                    "sensor": used_sensor,
                    **det,
                })
        else:
            rows.append({
                "aoi": aoi_name,
                "date": d,
                "vessel_count": 0,
                "sensor": used_sensor,
                "length_m": None,
                "width_m": None,
                "size_class": "none_detected",
                "centroid_xy": None,
                "angle_deg": None,
                "berth": None,
            })
    return pd.DataFrame(rows)

# Example run (uncomment once credentials are set):
#bg_dates = search_scenes(bboxes["zirku_spm_1"], "2026-01-01", "2026-03-01")[:8]
#df_a = build_timeseries("zirku_spm_1", "2026-06-01", "2026-07-31", bg_dates)
#df_b = build_timeseries("zirku_spm_2", "2026-06-01", "2026-07-31", bg_dates)
#df = pd.concat([df_a, df_b], ignore_index=True)
#df


def build_all_timeseries(aoi_names, start, end, background_dates_by_aoi, max_cloud=30):
    """
    aoi_names: list of keys from AOI_CONFIG
    background_dates_by_aoi: dict {aoi_name: [dates]} -- can reuse the same
        clear-period dates across AOIs in the same region if convenient,
        doesn't have to be per-AOI unique.
    """
    frames = []
    for name in aoi_names:
        bg_dates = background_dates_by_aoi[name]
        df = build_timeseries(name, start, end, bg_dates, max_cloud=max_cloud)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

from datetime import datetime

td = datetime.today().strftime('%Y-%m-%d')
TS_START = "2026-07-01"

# Background must be BEFORE the analysis window (exclude_dates drops any overlap).
bg_window = ("2026-06-01", "2026-06-30")
RUN_FULL_TIMESERIES = True  # True = fetch all AOIs/dates (~10+ min first run)

background_dates_by_aoi = {
    name: search_scenes(name, *bg_window)[:8] for name in LOADING_AOIS
}

if RUN_FULL_TIMESERIES:
    df_all = build_all_timeseries(LOADING_AOIS, TS_START, td, background_dates_by_aoi)
    df_all.to_csv("vessel_detections.csv", index=False)
    display(df_all)
else:
    df_all = None
    print("Skipped full timeseries — set RUN_FULL_TIMESERIES = True when ready.")

IndexError: list index out of range

## Pick berth slot boxes (interactive)

Defines `berth_slots_px` — pixel coords `[x1, y1, x2, y2]` on the fetched scene.

### Option A — Tkinter window (recommended in Cursor; no `%matplotlib` needed)

Run the **next code cell**. A separate desktop window opens.

- **Click-drag** to draw a box (size shown in the status bar)
- **Enter** — accept box
- **u** — undo last box
- **q** — quit

Yellow dashed = existing config boxes. Copy the printed list into the AOI config cell.

Yanbu has **4 berths** — set `PICK_AOI = "yanbu_north_crude_terminal"` (default). Muajjiz has 3.

### Option B — Terminal (if the notebook window does not appear)

```powershell
cd vessel_crossing\port_loading_gaps
..\.venv\Scripts\python pick_berths_standalone.py --aoi muajjiz --date 2026-08-01
```

### Option C — Matplotlib (Jupyter Lab only; often broken in Cursor)

Requires `pip install ipympl` and `%matplotlib widget` — use A or B instead.

In [12]:
# Berth box picker (Tkinter — works without %matplotlib widget/qt)
from berth_slot_picker import pick_berth_slots_tk, format_berth_slots_python

PICK_AOI = "das_spm_2"  # or "muajjiz"
PICK_DATE = "2026-07-31"
BERTH_VAR_NAMES = {
    "yanbu_north_crude_terminal": "YANBU_BERTH_SLOTS",
    "muajjiz": "MUAJJIZ_BERTH_SLOTS",
    "cpc": "CPC_BERTH_SLOTS",
    "das_spm_1": "DAS_SPM1_SLOTS",
    "das_spm_2": "DAS_SPM2_SLOTS",
    "zirku_spm_1": "ZIRKU_SPM1_SLOTS",
    "zirku_spm_2": "ZIRKU_SPM2_SLOTS",
    "um_lulu_spm": "UM_LULU_SPM_SLOTS",
}

pick_scene = fetch_scene(PICK_AOI, PICK_DATE)
existing = AOI_CONFIG[PICK_AOI].get("berth_slots_px", [])
slot_names = [s["name"] for s in existing] or None
var_name = BERTH_VAR_NAMES.get(PICK_AOI, "BERTH_SLOTS")

picked_slots = pick_berth_slots_tk(
    pick_scene,
    existing_slots=existing,
    slot_names=slot_names,
    var_name=var_name,
)

# Paste into AOI config, e.g. YANBU_BERTH_SLOTS = picked_slots
print(format_berth_slots_python(picked_slots, var_name=var_name))

IndexError: list index out of range

In [13]:
# --- Quick visual sanity check for one scene ------------------------------

def plot_detection(scene, detections, title=""):
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(scene)
    for det in detections:
        x, y = det["centroid_xy"]
        ax.scatter(x, y, s=80, facecolors="none", edgecolors="red", linewidths=2)
        ax.annotate(
            f"{det['size_class']} ({det['length_m']:.0f}m)",
            (x, y),
            color="red",
            fontsize=9,
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax.set_title(f"{title} — {len(detections)} vessel(s)")
    ax.axis("off")
    plt.show()


def plot_detection_debug(scene, background, detections, detect_params, title=""):
    """Scene / change mask / detections — use while tuning a new AOI."""
    mask, _ = build_change_mask(scene, background, detect_params)
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(scene)
    axes[0].set_title("Scene")
    axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Change mask")
    axes[1].axis("off")
    axes[2].imshow(scene)
    for det in detections:
        x, y = det["centroid_xy"]
        axes[2].scatter(x, y, s=80, facecolors="none", edgecolors="red", linewidths=2)
        axes[2].annotate(
            f"{det['size_class']} ({det['length_m']:.0f}m)",
            (x, y),
            color="red",
            fontsize=9,
            xytext=(5, 5),
            textcoords="offset points",
        )
    axes[2].set_title(f"{title} — {len(detections)} vessel(s)")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()


def plot_berth_slots(scene, detections, slots_px, title=""):
    """Draw berth slot boxes and mark occupied slots."""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(scene)
    occupied = {d.get("berth") for d in detections}
    for name, mask in build_berth_slot_masks(scene.shape, slots_px):
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        color = "lime" if name in occupied else "yellow"
        ax.plot([xs.min(), xs.max(), xs.max(), xs.min(), xs.min()],
                [ys.min(), ys.min(), ys.max(), ys.max(), ys.min()],
                color=color, linewidth=2)
        ax.text(xs.mean(), ys.min() - 3, name, color=color, fontsize=9, ha="center")
    for det in detections:
        x, y = det["centroid_xy"]
        ax.scatter(x, y, s=100, facecolors="none", edgecolors="red", linewidths=2)
    ax.set_title(f"{title} — {len(detections)} vessel(s)")
    ax.axis("off")
    plt.show()


# Example:
def plot_ndwi(ndwi, water_min=0.5, title="NDWI"):
    """Visual check — blue water should match Copernicus Browser NDWI layer."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    im = axes[0].imshow(ndwi, vmin=-0.3, vmax=0.8, cmap="RdYlBu")
    axes[0].set_title(f"{title} (raw)")
    plt.colorbar(im, ax=axes[0], fraction=0.046)
    wm = np.isfinite(ndwi) & (ndwi >= water_min)
    axes[1].imshow(wm, cmap="gray")
    axes[1].set_title(f"water mask (NDWI >= {water_min})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()


AOI = "zirku_spm_2"
SCENE_DATE = "2026-07-14"
params = detect_params_by_aoi[AOI]
scene, ndwi = _fetch_s2_rgb_ndwi(AOI, SCENE_DATE)
background = ensure_background(AOI)
dets = detect_vessels_for_aoi(AOI, scene, background, ndwi=ndwi)
# SPM tuning: per-slot signals (prelim vs final after relative filter)
# display(diagnose_berth_slots(scene, background, AOI_CONFIG[AOI]["berth_slots_px"], resolutions[AOI], params, ndwi=ndwi))
if ndwi is not None and params.get("use_ndwi_water"):
    plot_ndwi(ndwi, water_min=params.get("ndwi_water_min", 0.5), title=f"{AOI} {SCENE_DATE}")
if RESOLVED_AOI_CONFIG[AOI].get("mode") == "berth_slots":
    plot_berth_slots(scene, dets, AOI_CONFIG[AOI]["berth_slots_px"], title=f"{AOI} - {SCENE_DATE}")
else:
    plot_detection_debug(scene, background, dets, params, title=f"{AOI} - {SCENE_DATE}")
    plot_detection(scene, dets, title=f"{AOI} - {SCENE_DATE}")
dets

IndexError: list index out of range

## Join against Kpler port calls

Compare **occupancy counts** on each Sentinel scene date vs Kpler berthing intervals from the API (`loading_gaps.fetch_port_calls`).

| AOI | Kpler location | Notes |
|-----|----------------|-------|
| `yanbu_north_crude_terminal` | installation **Yanbu Crude** | |
| `muajjiz` | installation **Muajjiz** | |
| `zirku_spm_1`, `zirku_spm_2` | zone **Zirku** | One polygon per SPM; Kpler zone still aggregates both SPMs |
| `das_spm_1`, `das_spm_2` | zone **Das Island** | One polygon per SPM; Kpler zone aggregates both SPMs |
| `um_lulu_spm` | zone **Umm Lulu** | |

**Flags:** `agree` · `imagery_high` (likely FP or dark loading) · `imagery_low` (likely FN or timing). Use `count_tolerance=1` if ±1 vessel is acceptable.

In [ ]:
from datetime import date
from pathlib import Path

from loading_gaps import (
    DEFAULT_AOI_KPLER_LOCATIONS,
    LocationSpec,
    compare_imagery_with_kpler,
    fetch_port_calls_for_aois,
    parse_env,
    plot_imagery_kpler_comparison,
    summarize_imagery_kpler_comparison,
)

# AOI keys → Kpler installation or zone (see markdown above)
AOI_KPLER_LOCATIONS = {
    **DEFAULT_AOI_KPLER_LOCATIONS,
    # Per-SPM imagery AOIs share a Kpler zone when both SPMs sit in the same zone.
}

ENV_PATH = NOTEBOOK_DIR.parent / ".env"
KPLER_CRED = parse_env(ENV_PATH)
assert KPLER_CRED.get("KPLER_EMAIL") and KPLER_CRED.get("KPLER_PASSWORD"), (
    f"Missing KPLER_EMAIL / KPLER_PASSWORD in {ENV_PATH}"
)

# Use df_all from timeseries cell, or reload from CSV
if "df_all" not in globals() or df_all is None:
    df_all = pd.read_csv(NOTEBOOK_DIR / "vessel_detections.csv", parse_dates=["date"])

imagery_aois = sorted(df_all["aoi"].unique())
kpler_locations = {
    aoi: AOI_KPLER_LOCATIONS[aoi]
    for aoi in imagery_aois
    if aoi in AOI_KPLER_LOCATIONS
}
missing = set(imagery_aois) - set(kpler_locations)
if missing:
    print(f"No Kpler mapping for: {missing}")

kpler_start = pd.to_datetime(df_all["date"]).min().date()
kpler_end = pd.to_datetime(df_all["date"]).max().date()

port_calls_by_aoi = fetch_port_calls_for_aois(
    kpler_locations,
    start_date=kpler_start,
    end_date=kpler_end,
    email=KPLER_CRED["KPLER_EMAIL"],
    password=KPLER_CRED["KPLER_PASSWORD"],
)
for aoi, calls in port_calls_by_aoi.items():
    loc = kpler_locations[aoi]
    print(f"{aoi} ← Kpler {loc.kind} '{loc.name}': {len(calls):,} port calls")

COUNT_TOLERANCE = 0  # set to 1 to allow ±1 vessel vs Kpler
df_compare = compare_imagery_with_kpler(
    df_all,
    port_calls_by_aoi,
    aoi_locations=kpler_locations,
    count_tolerance=COUNT_TOLERANCE,
)
df_summary = summarize_imagery_kpler_comparison(df_compare)

display(df_summary)
display(
    df_compare.sort_values(["aoi", "date"]).style.apply(
        lambda row: [
            "background-color: #ffd6d6" if row["flag"] == "imagery_high"
            else "background-color: #d6e8ff" if row["flag"] == "imagery_low"
            else "background-color: #d6ffd6" if row["flag"] == "agree"
            else ""
        ]
        * len(row),
        axis=1,
    )
)

mismatches = df_compare[~df_compare["match"]].sort_values(["aoi", "date"])
print(f"Mismatches (|delta| > {COUNT_TOLERANCE}): {len(mismatches)}")
display(mismatches)

dark_loading_candidates = df_compare[
    (df_compare["imagery_count"] > 0) & (df_compare["kpler_count"] == 0)
]
if not dark_loading_candidates.empty:
    print("Dark-loading candidates (imagery > 0, Kpler = 0):")
    display(dark_loading_candidates)

# Overlay chart per terminal
for aoi in df_compare["aoi"].unique():
    plot_imagery_kpler_comparison(df_compare, aoi=aoi)
    plt.show()

## Notes / next steps

- **Revisit gaps**: Sentinel-2 gives ~3-5 day revisit and this is cloud-limited. Add a **Sentinel-1 SAR (CFAR detector)** leg for all-weather/day-night coverage — same AOIs, different evalscript/data collection (`DataCollection.SENTINEL1_IW`), and a CFAR threshold instead of background-diff.
- **Tuning**: `diff_thresh`, `min_area_px`, and the aspect-ratio cutoff will need tuning against a handful of known-good/known-bad scenes before trusting the size-class labels — sun glint and small clouds are the main false-positive sources in optical.
- **Fujairah anchorage specifically**: revisit whether background-diff detection is even the right tool there before trusting `dark_loadings`/`ghost_calls` output for it — it's a wide, busy holding area, not a discrete berth. You may want a lower-confidence "vessel density" metric for anchorages rather than per-vessel size classification.
- **AOI precision**: the point+buffer AOIs (Zirku, Yanbu, Fujairah terminal) are reasonable starting boxes but not verified against actual jetty/SPM footprints — worth eyeballing one Copernicus Browser scene per AOI to confirm the box actually frames the loading point before trusting outputs.
- **No Dagster yet**: this notebook is meant to be run manually or via a simple `cron`/Task Scheduler entry calling a `.py` version of `build_all_timeseries()` on a schedule, writing to CSV or straight to Snowflake with `snowflake-connector-python`. Easy to wire into Dagster later without changing the detection logic.
- **More locations**: add entries to `AOI_CONFIG` and `AOI_KPLER_LOCATIONS` (installation or zone) — fetch, detect, and Kpler compare are AOI-agnostic.